In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
df = pd.read_csv('Data/bank.csv', sep=';')
print("Shape:", df.shape)
print(df.head())

Shape: (4521, 17)
   age          job  marital  education default  balance housing loan  \
0   30   unemployed  married    primary      no     1787      no   no   
1   33     services  married  secondary      no     4789     yes  yes   
2   35   management   single   tertiary      no     1350     yes   no   
3   30   management  married   tertiary      no     1476     yes  yes   
4   59  blue-collar  married  secondary      no        0     yes   no   

    contact  day month  duration  campaign  pdays  previous poutcome   y  
0  cellular   19   oct        79         1     -1         0  unknown  no  
1  cellular   11   may       220         1    339         4  failure  no  
2  cellular   16   apr       185         1    330         1  failure  no  
3   unknown    3   jun       199         4     -1         0  unknown  no  
4   unknown    5   may       226         1     -1         0  unknown  no  


In [34]:
print("Duplicate Rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)


Duplicate Rows: 0


In [35]:
print(df.isnull().sum())


age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [36]:
#Check unknown values
for col in df.select_dtypes(include=['object','string']).columns:
    unknown_count = (df[col] == 'unknown').sum()
    percent = (unknown_count / len(df)) * 100
    print(f"{col} → {unknown_count} ({percent:.2f}%) unknown")



job → 38 (0.84%) unknown
marital → 0 (0.00%) unknown
education → 187 (4.14%) unknown
default → 0 (0.00%) unknown
housing → 0 (0.00%) unknown
loan → 0 (0.00%) unknown
contact → 1324 (29.29%) unknown
month → 0 (0.00%) unknown
poutcome → 3705 (81.95%) unknown
y → 0 (0.00%) unknown


In [37]:
#TARGET & BINARY CONVERSION

df['y'] = df['y'].map({'yes':1,'no':0})

binary_cols = ['default','housing','loan']
for col in binary_cols:
    df[col] = df[col].map({'yes':1,'no':0})



In [38]:
# FEATURE ENGINEERING
df['previous_contacted'] = df['poutcome'].apply(
    lambda x: 0 if x == 'unknown' else 1
)
df['previous_success'] = df['poutcome'].apply(
    lambda x: 1 if x == 'success' else 0
)
df['age_group'] = pd.cut(df['age'],
                         bins=[18,30,45,60,100],
                         labels=['Young','Adult','Senior','Old'])
df.drop('poutcome', axis=1, inplace=True)


In [39]:
print("Min balance:", df['balance'].min())
print("Min duration:", df['duration'].min())



Min balance: -3313
Min duration: 4


In [40]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='yeo-johnson')

df[['balance','duration']] = pt.fit_transform(df[['balance','duration']])


In [41]:
print("New Min balance:", df['balance'].min())
print("New Min duration:", df['duration'].min())

print("Any NaN:", df.isnull().sum().sum())

print("Any Inf:",
      np.isinf(df.select_dtypes(include=[np.number])).sum().sum())



New Min balance: -13.487834983619454
New Min duration: -3.0961207802707977
Any NaN: 0
Any Inf: 0


In [42]:
# Save preprocessed dataset
df.to_csv("Data/preprocessed_data.csv", index=False)

In [43]:
#Check unknown values
for col in df.select_dtypes(include=['object','string']).columns:
    unknown_count = (df[col] == 'unknown').sum()
    percent = (unknown_count / len(df)) * 100
    print(f"{col} → {unknown_count} ({percent:.2f}%) unknown")

job → 38 (0.84%) unknown
marital → 0 (0.00%) unknown
education → 187 (4.14%) unknown
contact → 1324 (29.29%) unknown
month → 0 (0.00%) unknown


In [44]:
print("Shape:", df.shape)
print("Any NaN:", df.isnull().sum().sum())
print("Any Inf:",
      np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

print("Remaining categorical columns:",
      df.select_dtypes(include=['object','string']).columns)

Shape: (4521, 19)
Any NaN: 0
Any Inf: 0
Remaining categorical columns: Index(['job', 'marital', 'education', 'contact', 'month'], dtype='str')
